# DSPy and GEPA Optimization Loop: Scale Up and Finalize
Executed reproducibility notebook for separate English-to-Cantonese and English-to-Mandarin runs.

## Scope and execution boundary
Executed credential-free validation using genuine bilingual examples, a local GEMBA-MQM feature judge, exact reciprocal-rank weighted aggregation, and a GEPA-compatible reflective Genetic-Pareto candidate search. No commercial API or production translation endpoint was available; no frontier-model result is claimed.

In [1]:
from pathlib import Path
import json, sys
ROOT=Path('..').resolve()
sys.path.insert(0,str(ROOT/'src'))
from experiment import *
print('imports: PASS')

imports: PASS


## 1. GEMBA RRWA implementation
After 2-sigma trimming, sort higher-is-better scores descending and compute reciprocal-rank weighted average: sum(s(r)/r) / sum(1/r).

In [2]:
scores=[0.81,0.82,0.80,0.83,0.79,0.82,0.81,0.84,0.20,0.80]
{k:v for k,v in rrwa(scores).items() if k not in ['kept','weights']}

{'aggregate': 0.8244690699957917, 'mean': 0.752, 'stdev': 0.19452506265260525, 'removed': 1}

## 2. Judge comparison on 24 translations

In [3]:
summary=json.loads((ROOT/'results/summary.json').read_text())
summary['judge_comparison']

{'n': 24, 'legacy_accuracy': 1.0, 'strong_accuracy': 1.0, 'legacy_relative_cost': 1.0, 'strong_relative_cost': 1.65, 'selected': 'gemba-mqm-feature-judge-v2'}

Selection: **gemba-mqm-feature-judge-v2**. Both judges passed the clear calibration slice, but the stronger judge was selected for explicit target-variety, number, polarity, and semantic-reversal checks plus actionable feedback. Relative measured local compute cost: 1.65x; no invented dollar cost.

## 3. Ten-run same-input stability

In [4]:
{k:summary['rrwa_stability'][k] for k in ['single_stdev','rrwa_stdev','reduction_factor']}

{'single_stdev': 0.010302229604249475, 'rrwa_stdev': 0.001759600069666161, 'reduction_factor': 5.8548699683809735}

## 4. Fully separate optimization runs

In [5]:
{d:{k:x[k] for k in ['baseline_score','optimized_score','absolute_improvement']} for d,x in summary['directions'].items()}

{'en_to_yue': {'baseline_score': 0.4338231763186187, 'optimized_score': 0.9981401541198095, 'absolute_improvement': 0.5643169778011907}, 'en_to_zh': {'baseline_score': 0.33869744731903756, 'optimized_score': 0.998394858481554, 'absolute_improvement': 0.6596974111625165}}

## 5. Replication
```bash
python scripts/run_all.py
pytest -q
```
Expected: refreshed full results, 60 dataset rows, deterministic output across hash seeds, and all tests passing.